In [153]:
%matplotlib inline
from __future__ import division
import matplotlib
import numpy as np
from pylab import *
import os
matplotlib.rcParams.update({"axes.formatter.limits": (-4,4)})
plotStyles={"markersize":12,"markeredgewidth":3.0,"linewidth":3.0}
stepStyles={"markersize":12,"markeredgewidth":3.0,"linewidth":3.0,"where":"post"}
np.seterr(divide='ignore',invalid='ignore')
pass

### Setup the notebook.

In [154]:
import h5py
import math
testNames=["ligand_receptor",
           "ligand_receptor_multiple",
           "ligand_receptor_phospho_state",
           "phosphorylation_state",
           "ste5_activity"]
test_names_bash_list=" ".join(testNames)
def isclose(a, b, rel_tol=1e-09, abs_tol=0.0):
    return abs(a-b) <= max(rel_tol * max(abs(a), abs(b)), abs_tol)

In [155]:
%%bash
rm -rf tmp && mkdir tmp

### Execute the BNGL imports.

In [156]:
%%bash -s "$test_names_bash_list"
for testName in $1; do
    inputFilename=${testName}.bngl
    outputFilename=tmp/${testName}.lm
    rm -f ${outputFilename}* && lm_bionetgen_import --verbose ${outputFilename} ${inputFilename} 2>&1 > ${outputFilename}.log
done;
echo "Finished."

Finished.


### Define the tests.

In [157]:
def test_ligand_receptor(testOutputFilename):
    fp = h5py.File(testOutputFilename, "r")
    nr=fp["/Model/Reaction/"].attrs["numberReactions"]
    ns=fp["/Model/Reaction/"].attrs["numberSpecies"]
    D=fp["/Model/Reaction/DependencyMatrix"]
    C=fp["/Model/Reaction/InitialSpeciesCounts"]
    K=fp["/Model/Reaction/ReactionRateConstants"]
    R=fp["/Model/Reaction/ReactionTypes"]
    S=fp["/Model/Reaction/StoichiometricMatrix"]
    if nr != 2: raise Exception("numberReactions: incorrect value")
    if ns != 3: raise Exception("numberSpecies: incorrect value")
    if D.shape != (ns,nr): raise Exception("D: incorrect shape",D.shape)
    if np.any(D[:,0] != np.array([1,1,0])): raise Exception("D: incorrect value")
    if np.any(D[:,1] != np.array([0,0,1])): raise Exception("D: incorrect value")
    if C.shape != (ns,): raise Exception("C: incorrect shape")
    if np.any(C[:] != np.array([602000,10000,0])): raise Exception("C: incorrect values")
    if K.shape != (nr,10): raise Exception("K: incorrect shape")
    if not isclose(K[0,0],1.66e-7,1e-2): raise Exception("K: incorrect value:",K[0,0])
    if not isclose(K[1,0],0.1,1e-4): raise Exception("K: incorrect value:",K[1,0])
    for i in range(0,nr):
        for j in range(1,10):
            if not math.isnan(K[i,j]): raise Exception("K: incorrect nan value")
    if R.shape != (nr,): raise Exception("R: incorrect shape")
    if np.any(R[:] != np.array([2,1])): raise Exception("R: incorrect valuess")
    if S.shape != (ns,nr): raise Exception("S: incorrect shape")
    if np.any(S[:,0] != np.array([-1,-1,1])): raise Exception("S: incorrect value")
    if np.any(S[:,1] != np.array([1,1,-1])): raise Exception("S: incorrect value")
    fp.close()

In [158]:
def test_ligand_receptor_multiple(testOutputFilename):
    fp = h5py.File(testOutputFilename, "r")
    nr=fp["/Model/Reaction/"].attrs["numberReactions"]
    ns=fp["/Model/Reaction/"].attrs["numberSpecies"]
    D=fp["/Model/Reaction/DependencyMatrix"]
    C=fp["/Model/Reaction/InitialSpeciesCounts"]
    K=fp["/Model/Reaction/ReactionRateConstants"]
    R=fp["/Model/Reaction/ReactionTypes"]
    S=fp["/Model/Reaction/StoichiometricMatrix"]
    if nr != 8: raise Exception("numberReactions: incorrect value",nr)
    if ns != 5: raise Exception("numberSpecies: incorrect value",ns)
    if D.shape != (ns,nr): raise Exception("D: incorrect shape",D.shape)
    if np.sum(D[0,:]) != 4: raise Exception("D: incorrect value")
    for i in range (1,ns):
        if np.sum(D[i,:]) != 2: raise Exception("S: incorrect value")
    if C.shape != (ns,): raise Exception("C: incorrect shape")
    if np.any(C[:] != np.array([602000,10000,0,0,0])): raise Exception("C: incorrect values")
    if K.shape != (nr,10): raise Exception("K: incorrect shape")
    for i in range (0,nr):
        if not isclose(K[i,0],1.66e-7,1e-2) and not isclose(K[i,0],0.1,1e-2): raise Exception("K: incorrect value:")
    for i in range(0,nr):
        for j in range(1,10):
            if not math.isnan(K[i,j]): raise Exception("K: incorrect nan value")
    if R.shape != (nr,): raise Exception("R: incorrect shape")
    for i in range (0,nr):
        if R[i] != 1 and R[i] != 2: raise Exception("R: incorrect values")
    if S.shape != (ns,nr): raise Exception("S: incorrect shape")
    if np.sum(S[0,:]) != 0: raise Exception("S: incorrect value")
    if np.sum(abs(S[0,:])) != 8: raise Exception("S: incorrect value")
    for i in range (1,ns):
        if np.sum(S[i,:]) != 0: raise Exception("S: incorrect value")
        if np.sum(abs(S[i,:])) != 4: raise Exception("S: incorrect value")
    fp.close()

In [166]:
def test_ligand_receptor_phospho_state(testOutputFilename):
    fp = h5py.File(testOutputFilename, "r")
    nr=fp["/Model/Reaction/"].attrs["numberReactions"]
    ns=fp["/Model/Reaction/"].attrs["numberSpecies"]
    D=fp["/Model/Reaction/DependencyMatrix"]
    C=fp["/Model/Reaction/InitialSpeciesCounts"]
    K=fp["/Model/Reaction/ReactionRateConstants"]
    R=fp["/Model/Reaction/ReactionTypes"]
    S=fp["/Model/Reaction/StoichiometricMatrix"]
    if nr != 8: raise Exception("numberReactions: incorrect value",nr)
    if ns != 5: raise Exception("numberSpecies: incorrect value",ns)
    if D.shape != (ns,nr): raise Exception("D: incorrect shape",D.shape)
    for i in range (0,ns):
        if np.sum(D[i,:]) != 2: raise Exception("S: incorrect value")
    if C.shape != (ns,): raise Exception("C: incorrect shape")
    if np.any(C[:] != np.array([100,10,0,0,0])): raise Exception("C: incorrect values")
    if K.shape != (nr,10): raise Exception("K: incorrect shape")
    for i in range(0,nr):
        for j in range(1,10):
            if not math.isnan(K[i,j]): raise Exception("K: incorrect nan value")
    if R.shape != (nr,): raise Exception("R: incorrect shape")
    for i in range (0,nr):
        if R[i] != 1 and R[i] != 2: raise Exception("R: incorrect values")
    if S.shape != (ns,nr): raise Exception("S: incorrect shape")
    for i in range (0,ns):
        if np.sum(S[i,:]) != 0: raise Exception("S: incorrect value")
        if np.sum(abs(S[i,:])) != 4: raise Exception("S: incorrect value")
    fp.close()

In [167]:
def test_phosphorylation_state(testOutputFilename):
    fp = h5py.File(testOutputFilename, "r")
    nr=fp["/Model/Reaction/"].attrs["numberReactions"]
    ns=fp["/Model/Reaction/"].attrs["numberSpecies"]
    D=fp["/Model/Reaction/DependencyMatrix"]
    C=fp["/Model/Reaction/InitialSpeciesCounts"]
    K=fp["/Model/Reaction/ReactionRateConstants"]
    R=fp["/Model/Reaction/ReactionTypes"]
    S=fp["/Model/Reaction/StoichiometricMatrix"]
    if nr != 8: raise Exception("numberReactions: incorrect value",nr)
    if ns != 4: raise Exception("numberSpecies: incorrect value",ns)
    if D.shape != (ns,nr): raise Exception("D: incorrect shape",D.shape)
    for i in range (0,ns):
        if np.sum(D[i,:]) != 2: raise Exception("D: incorrect value")
    if C.shape != (ns,): raise Exception("C: incorrect shape")
    if np.any(C[:] != np.array([10,0,0,0])): raise Exception("C: incorrect values")
    if K.shape != (nr,10): raise Exception("K: incorrect shape")
    for i in range (0,nr):
        if not isclose(K[i,0],2.0,1e-2) and not isclose(K[i,0],3.0,1e-2) and not isclose(K[i,0],4.0,1e-2) and not isclose(K[i,0],5.0,1e-2):
            raise Exception("K: incorrect value:")
    for i in range(0,nr):
        for j in range(1,10):
            if not math.isnan(K[i,j]): raise Exception("K: incorrect nan value")
    if R.shape != (nr,): raise Exception("R: incorrect shape")
    for i in range (0,nr):
        if R[i] != 1: raise Exception("R: incorrect values")
    if S.shape != (ns,nr): raise Exception("S: incorrect shape")
    if np.sum(S[0,:]) != 0: raise Exception("S: incorrect value")
    for i in range (0,ns):
        if np.sum(S[i,:]) != 0: raise Exception("S: incorrect value")
        if np.sum(abs(S[i,:])) != 4: raise Exception("S: incorrect value")
    fp.close()

In [168]:
def test_ste5_activity(testOutputFilename):
    fp = h5py.File(testOutputFilename, "r")
    nr=fp["/Model/Reaction/"].attrs["numberReactions"]
    ns=fp["/Model/Reaction/"].attrs["numberSpecies"]
    D=fp["/Model/Reaction/DependencyMatrix"]
    C=fp["/Model/Reaction/InitialSpeciesCounts"]
    K=fp["/Model/Reaction/ReactionRateConstants"]
    R=fp["/Model/Reaction/ReactionTypes"]
    S=fp["/Model/Reaction/StoichiometricMatrix"]
    if nr != 896: raise Exception("numberReactions: incorrect value",nr) # 384+64*4*2
    if ns != 131: raise Exception("numberSpecies: incorrect value",ns)
    if D.shape != (ns,nr): raise Exception("D: incorrect shape",D.shape)
    if np.sum(D[0,:]) != 7: raise Exception("D: incorrect value")
    if np.sum(D[1,:]) != 64: raise Exception("D: incorrect value")
    if np.sum(D[2,:]) != 64: raise Exception("D: incorrect value")
    if np.sum(D[3,:]) != 64: raise Exception("D: incorrect value")
    for i in range (4,ns):
        if np.sum(D[i,:]) != 7: raise Exception("D: incorrect value")
    if C.shape != (ns,): raise Exception("C: incorrect shape")
    if np.any(C[0:4] != np.array([2,3,4,5])): raise Exception("C: incorrect values")
    for i in range (4,ns):
        if C[i] != 0: raise Exception("C: incorrect values")
    if K.shape != (nr,10): raise Exception("K: incorrect shape")
    for i in range (0,nr):
        if not isclose(K[i,0],1.0,1e-2): raise Exception("K: incorrect value:")
    for i in range(0,nr):
        for j in range(1,10):
            if not math.isnan(K[i,j]): raise Exception("K: incorrect nan value")
    if R.shape != (nr,): raise Exception("R: incorrect shape")
    for i in range (0,nr):
        if R[i] != 1 and R[i] != 2: raise Exception("R: incorrect values")
    if S.shape != (ns,nr): raise Exception("S: incorrect shape")
    if np.sum(S[0,:]) != 0: raise Exception("S: incorrect value")
    for i in range (0,ns):
        if np.sum(S[i,:]) != 0: raise Exception("S: incorrect value")
    if np.sum(abs(S[0,:])) != 14: raise Exception("S: incorrect value")
    if np.sum(abs(S[1,:])) != 128: raise Exception("S: incorrect value")
    if np.sum(abs(S[2,:])) != 128: raise Exception("S: incorrect value")
    if np.sum(abs(S[3,:])) != 128: raise Exception("S: incorrect value")
    for i in range (4,ns):
        if np.sum(abs(S[i,:])) != 14: raise Exception("S: incorrect value")
    fp.close()

### Run the tests.

In [169]:
testMethods = {"ligand_receptor": test_ligand_receptor,
              "ligand_receptor_multiple": test_ligand_receptor_multiple,
              "ligand_receptor_phospho_state": test_ligand_receptor_phospho_state,
              "phosphorylation_state": test_phosphorylation_state,
              "ste5_activity": test_ste5_activity}
for testName in testNames:
    try:
        testOutputFilename="tmp/%s.lm"%(testName)
        testMethods[testName](testOutputFilename)
    except Exception as e:
        print "%-60s : FAILED with:"%("["+testName+"]"),e
    except:
        print "%-60s : FAILED with: Unknown exception"%("["+testName+"]")
    else:
        print "%-60s : passed."%("["+testName+"]")

[ligand_receptor]                                            : passed.
[ligand_receptor_multiple]                                   : passed.
[ligand_receptor_phospho_state]                              : passed.
[phosphorylation_state]                                      : passed.
[ste5_activity]                                              : passed.


In [170]:
%%bash
rm -rf tmp